# Laboratorio 12 — El panel del administrador

**Duración: 45 minutos.**

Hoy no hay conceptos nuevos. Hoy se arma la herramienta.

Durante once laboratorios miraste las vistas del sistema de a una, para entender algo. Hoy
las vas a usar juntas, como las usa quien administra la plataforma: **ocho preguntas que se
corren cada mañana** para saber si las tablas están sanas antes de que alguien reclame.

Este cuaderno es lo que te llevas al trabajo. Guárdalo.

---

### Lo primero, y lo que hace posible todo lo demás

**Ninguna de las ocho consultas lee datos.** Todas leen lo que la libreta tiene anotado
de sí misma —cuántos papelitos hay, cuánto pesa cada uno, qué se escribió y cuándo—, que
es poquísimo y está a mano. Por eso se pueden correr **todas las mañanas sobre todas las
tablas** sin que a nadie le moleste.

Si tuvieras que abrir los archivos para responderlas, esto sería un informe mensual. Como
no hace falta, es un panel.

---

### Antes de empezar

Las ocho consultas de hoy miran **una sola tabla, `panel_lab12`**, y el paso 0 la arma.

No la vas a cargar como se carga una tabla sana. La vas a cargar **con los achaques
puestos a propósito**: un mes enorme y los demás flacos, muchas escrituras chicas,
correcciones que dejan papeles viejos en el suelo, una columna agregada a mitad de camino
y un borrado al final. Cada achaque es lo que va a encontrar una de las ocho preguntas.

Eso tiene dos consecuencias buenas. La primera es que **toda la sala ve los mismos
números**, así que se puede discutir la pantalla en voz alta. La segunda es que cada
pregunta encuentra algo: un panel que responde "todo bien" ocho veces no enseña a leerlo.

Y son cortas a propósito: cuatro o cinco líneas cada una. Al final del paso 1 vas a ver
cómo se extienden a todas tus tablas de una vez, que es la forma que toman en producción.


## Paso 0 — La tabla de hoy, con sus achaques

Primero, a tu espacio de trabajo.

In [ ]:
%%sql
-- Celda 0.1



Partimos de cero, por si ya habías corrido este laboratorio antes.

In [ ]:
%%sql
-- Celda 0.2



**La carga inicial: todo enero de un tirón.**

Así empiezan casi todas las tablas de una institución. Alguien trae el histórico que ya
existía y lo mete de una vez, y esa primera carga queda mucho más gorda que todo lo que
llega después de a poco.

La tabla va **particionada por mes**: `PARTITIONED BY (months(fecha_emision))` le dice a
la libreta que guarde en un cajón distinto los documentos de cada mes. Y `AS SELECT` crea
la tabla y la llena con el resultado de la consulta, todo en una sentencia.

Esta es **la partición grande**, y en el paso 5 va a saltar a la vista.

In [ ]:
%%sql
-- Celda 0.3



Enero adentro. Ahora **los once meses siguientes, pero como llegan de verdad**: de a
poquito, un día a la vez.

Esta celda **ya viene escrita** y es la única del cuaderno que no es SQL. No hay que
cambiarle nada. Viene escrita porque lo que hace son **veintidós cargas**, una por día,
y teclear veintidós sentencias iguales no enseña nada: lo que importa es lo que deja, no
el tecleo.

Lee el ciclo antes de correrla, que se entiende solo: para cada mes de febrero a
diciembre, y para el día 10 y el día 20 de ese mes, mete en la tabla los documentos de
ese día. Es una **ingesta diaria** en miniatura, que es como se cargan las tablas de
verdad: un poco cada día, todos los días.

Demora más que las otras celdas, porque son veintidós escrituras seguidas.

In [ ]:
# Celda 0.4
ESPACIO = "mi_espacio"

for mes in range(2, 13):
    for dia in (10, 20):
        fecha = f"2024-{mes:02d}-{dia:02d}"
        spark.sql(
            f"INSERT INTO {ESPACIO}.panel_lab12 "
            f"SELECT * FROM curso.dte_2024 WHERE fecha_emision = DATE '{fecha}'"
        )
        print("cargado", fecha)

print("listo: 22 cargas diarias")

Veintidós cargas, veintidós anotaciones en la libreta.

Ahora **tres correcciones sobre enero**, de esas que pasan cuando el histórico venía con
errores y hay que arreglarlo una vez cargado.

La primera marca como aceptados los documentos de enero que quedaron en revisión. Fíjate
en lo que esto le cuesta a la libreta: para cambiar unas filas de un papelito, Iceberg
**no puede tachar** —los papelitos no se corrigen— así que **escribe un papelito nuevo
con el contenido ya corregido** y deja de usar el viejo. El viejo queda en el suelo, y en
el paso 4 lo vas a contar.

In [ ]:
%%sql
-- Celda 0.5



La segunda corrección: a los documentos del 5 de enero les faltaba el IVA.

In [ ]:
%%sql
-- Celda 0.6



Y la tercera: el primer día del año quedó con el tipo de documento equivocado.

In [ ]:
%%sql
-- Celda 0.7



Tres correcciones, tres papelitos viejos en el suelo.

Ahora **una columna nueva**. Aparece una exigencia que no existía cuando se creó la
tabla: hay que dejar registrado quién revisó cada documento.

`ADD COLUMN` no toca ni un dato: cambia **la estructura**, que es otra cosa. Los
documentos que ya estaban quedan con esa columna vacía, y desde ahora la tabla tiene una
columna más. En el paso 7 vas a ver que la libreta se acuerda de este momento.

In [ ]:
%%sql
-- Celda 0.8



Y lo último, **un borrado**.

Se detectó que los documentos del 2024-07-20 se habían cargado dos veces, así que hay
que sacarlos. Es un día completo, uno de los veintidós que cargaste recién.

Eso tiene una consecuencia que importa para el paso 6: como ese día entró **en una sola
carga**, sus documentos están todos en el mismo papelito y en ningún otro. Al borrarlos,
la libreta no necesita escribir nada nuevo, le basta con **dejar de nombrar ese
papelito**. Por eso esta anotación va a quedar registrada como un borrado y no como una
reescritura.

In [ ]:
%%sql
-- Celda 0.9



Listo. La tabla quedó **con todos sus achaques puestos**: un mes gordo y once flacos,
veintitantas escrituras, papelitos viejos en el suelo, una columna agregada a mitad de
camino y un borrado al final.

Así es como llega al panel una tabla que lleva un tiempo en producción. Ahora empiezan
las ocho preguntas.

## Paso 1 — ¿Cuántos archivos tiene la tabla y cuánto pesan?

**Primera pregunta del panel, y es la de siempre:** cuánto ocupa la tabla y en cuántos
pedazos está.

Muchos archivos para pocos datos es la señal de que hace falta compactar. La tabla que
acabas de armar tiene veintitantas escrituras encima, así que ya sabes por dónde va a
salir.

In [ ]:
%%sql
-- Celda 1.1



Esa es la consulta, y así de corta se queda. Pero un panel no mira una tabla: mira
todas. Esta es la forma que va a tener en tu plataforma, y la vas a correr.

Se agrega una columna con el nombre de la tabla, se repite el bloque por cada una y se
unen con `UNION ALL`. Aquí van dos, la de hoy y otra tuya; en producción esto no se
teclea a mano, lo arma un script que recorre las tablas y repite el bloque por cada una
que encuentra.

Fíjate de paso en el precio que tiene: si **una sola** de las tablas nombradas no
existiera, falla la consulta entera y no devuelve ninguna fila, ni siquiera la de las
tablas que sí están. Por eso el script que la arma tiene que leer la lista de tablas de
verdad y no traerla escrita.

In [ ]:
%%sql
-- Celda 1.2



## Paso 2 — ¿Cuántos archivos chicos hay?

**Segunda pregunta:** cuántos de esos archivos son *chicos*.

Un archivo chico cuesta casi lo mismo de abrir que uno grande: hay que ir a buscarlo,
abrirlo y leer su encabezado antes de mirar una sola fila. Mil archivos de diez
kilobytes son mil aperturas para leer diez megabytes, y ahí se va el tiempo de las
consultas.

La cura es juntarlos, pasarlos en limpio a unas pocas hojas. Esta pregunta no los
junta: los **cuenta**, para saber si hace falta.

El umbral de «chico» depende de la plataforma. Aquí usamos un megabyte, que en el
laboratorio es razonable; en el SII, con archivos de cientos de megas, el umbral sería más
alto.

In [ ]:
%%sql
-- Celda 2.1



## Paso 3 — ¿Cuántas páginas acumula la tabla?

**Tercera pregunta:** cuánta historia arrastra la tabla, y desde cuándo.

La fecha de la página más vieja es la que importa: dice hasta dónde hacia atrás se puede
preguntar hoy, y también cuánta historia habría que botar si el disco aprieta.

In [ ]:
%%sql
-- Celda 3.1



## Paso 4 — ¿Cuánto ocupan las páginas que ya no cuentan?

**Cuarta pregunta:** cuánto espacio están ocupando los archivos que ya nadie
consulta.

Cada vez que se corrige algo, la libreta **no tacha**: escribe un papelito nuevo con el
contenido corregido y deja de nombrar el viejo. El viejo no se borra, sigue en el disco,
porque las páginas anteriores todavía lo nombran y mientras existan se puede pedir la
tabla como estaba antes.

La cuenta es la diferencia entre `.all_files`, que son todos los que hay en el disco, y
`.files`, que son los que cuentan hoy. Esa diferencia es lo que se liberaría al botar
las páginas viejas.

In [ ]:
%%sql
-- Celda 4.1



## Paso 5 — ¿Hay particiones desparejas?

**Quinta pregunta**, y esta solo aplica a tablas particionadas. Por eso hoy la tabla es
`panel_lab12` y no otra.

Una partición desbalanceada es una que tiene muchas más filas que las demás. Duele por dos
razones: la consulta que cae en ella tarda mucho más que las otras, y cuando hay que
reprocesarla, el trabajo no se reparte.

La vista `.partitions` responde en una línea.

**Un aviso sobre la primera columna, para que no te desconcierte.** No dice `2024-01`:
dice un número, `648`. Es **cuántos meses pasaron desde enero de 1970**, que es como
Iceberg guarda por dentro un cajón mensual. Seiscientos cuarenta y ocho meses son
cincuenta y cuatro años justos, así que `648` es enero de 2024, `649` es febrero, y así.
No hace falta traducirlo para leer la salida: lo que importa es **comparar las filas de
una fila con las de las otras**, y el orden ya te pone arriba la más gorda.

In [ ]:
%%sql
-- Celda 5.1



## Paso 6 — ¿Cuándo fue la última escritura?

**Sexta pregunta:** cuándo escribió alguien por última vez en la tabla, y qué hizo.

No es para vigilar a nadie. Sirve para tres cosas, y la primera es la que más se usa.

**Uno, la carga que no llegó.** Si sabes cada cuánto debería cargarse la tabla —todas las
noches, todos los lunes, el día 5 de cada mes— entonces la última escritura te dice al
tiro si la carga de anoche entró. Una tabla que debía cargarse ayer y tiene su última
anotación de hace cuatro días es una carga caída, y el panel la muestra antes de que
alguien reclame por un reporte incompleto.

**Dos, la operación que no corresponde.** La columna `operation` dice qué se hizo. Un
`delete` en una tabla en la que solo se supone que se agregan documentos es algo que hay
que ir a preguntar hoy, no el mes que viene.

**Tres, la tabla muerta.** Si no hay escrituras hace meses y nadie la reclama, es una
tabla que sigue ocupando disco y saliendo en los respaldos sin que le sirva a nadie.

In [ ]:
%%sql
-- Celda 6.1



## Paso 7 — ¿Cuántas versiones de portada hay?

**Séptima pregunta**, y usa una vista que no habías visto: `.metadata_log_entries`.

Cada vez que algo cambia en la tabla se escribe un `metadata.json` nuevo, la portada de la
libreta. Esa vista los lista todos, con el esquema que estaba vigente en cada uno.

Que el `latest_schema_id` haya cambiado significa que **alguien tocó la estructura**:
agregó una columna, la renombró o cambió un tipo. Es lo que uno quiere saber cuando algo
que funcionaba deja de funcionar.

In [ ]:
%%sql
-- Celda 7.1



## Paso 8 — ¿La tabla se compactó alguna vez?

**Octava y última pregunta**, y es la que cierra el círculo.

`rewrite_data_files` deja siempre una página de tipo `replace`. Así que una tabla con
muchos archivos y **ninguna** página `replace` es una tabla que nadie ha compactado nunca.

Esta consulta responde las dos cosas de una vez.

In [ ]:
%%sql
-- Celda 8.1



## Cierre

Las ocho preguntas, con un umbral sugerido para cada una. Ajústalos a tu plataforma: estos
son los del laboratorio.

| # | La pregunta | Cuándo preocuparse |
|---|---|---|
| 1 | Archivos y peso por tabla | muchos archivos para pocas filas |
| 2 | Archivos chicos | más de la mitad bajo el umbral |
| 3 | Páginas acumuladas | más de cien, o la más vieja de hace meses |
| 4 | Espacio de páginas muertas | si pasa del doble de lo vigente |
| 5 | Particiones desparejas | si una tiene diez veces más filas que la mediana |
| 6 | Última escritura | si no hay ninguna hace meses, la tabla puede estar muerta |
| 7 | Versiones de portada | si el esquema cambió y nadie avisó |
| 8 | Tablas sin compactar | muchos archivos y cero `replace` |

**Y una advertencia sobre esa última columna, porque es la que más se malinterpreta.**
Ninguno de esos umbrales es una regla de Iceberg. Iceberg no dice en ninguna parte que
cien páginas sean muchas ni que un megabyte sea chico: son los números de este
laboratorio, puestos para que se pueda conversar.

Cada plataforma fija los suyos, y los fija mirando lo suyo: el tamaño de sus tablas, cada
cuánto se cargan, cuánta historia exige conservar su auditoría y cuánto disco tiene. Un
archivo de un megabyte es chico aquí y sería ridículamente chico en una tabla de cientos
de gigas, donde el umbral anda en cientos de megas. La primera tarea del que arma este
panel en su institución es reemplazar esa columna por la de su casa.

Y la frase con la que hay que quedarse:

> **Ninguna de estas consultas lee datos.** Todas leen lo que la libreta tiene anotado de
> sí misma. Por eso se pueden correr todas las mañanas sobre todas las tablas.

Eso es lo que separa a este panel de un informe. Un informe se pide; un panel se mira. Y lo
que se mira todos los días es lo que evita la llamada del viernes a las seis de la tarde.
